# 01 — Data Audit, Filename Reconstruction and Evaluation Splits

**Research question:** What do the image filenames actually encode, and how can those fields be used to build four independently validated train/validation/held-out-test designs?

This notebook **does not assume** that source IDs or chronological metadata were supplied in a separate dataset table. It inspects each class-labelled image filename, extracts environment/video IDs and frame numbers, audits collisions, and then builds deterministic splits. The first code cell sets `SMOKE_TEST` from the `MONREADER_SMOKE_TEST` environment variable. A researcher normally leaves it unset (full research profile). The smoke profile is strictly for testing with synthetic data; its metrics are not project findings.

**Repository layout:** the only required tracked files are the five notebooks, README and the original dataset. Notebook 01 generates all shared Python methods locally under `generated/`; this file is constructed from the following fully visible source cell, so no external hidden `src/` package is required.

In [ ]:
from pathlib import Path
import os, json, sys
ROOT = Path.cwd().resolve()
SMOKE_TEST = os.getenv('MONREADER_SMOKE_TEST', '0') == '1'
GENERATED = ROOT / 'generated'
GENERATED.mkdir(exist_ok=True)
print('Notebook root:', ROOT)
print('Dataset root:', os.getenv('MONREADER_DATA_ROOT', str(ROOT/'data'/'raw')))
print('Execution profile:', 'SYNTHETIC SMOKE' if SMOKE_TEST else 'REAL DATA RESEARCH')

## Reusable methods, defined and generated from this notebook

The next code cell implements the filename parser, source-aware splits, models, Bayesian search, held-out predictions, source bootstrap, and Grad-CAM. The source is written to a generated `.py` file to allow notebooks 02–05 to reuse exactly the same functions in separate kernels. Nothing in these functions contains a hard-coded historical answer; mathematical or leakage assertions check only structural integrity.

In [ ]:
%%writefile generated/monreader_methods.py
"""MonReader methods. This file is generated from the source code embedded in Notebook 01.
It is NOT a separately required repository dependency.

Research principle: all data-derived quantities are calculated from the active run.
No historical metric or figure is treated as an expected answer.
"""
from __future__ import annotations
import copy, hashlib, json, os, random, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score, average_precision_score

REGIMES = ('O', 'S', 'T', 'ST')
CLASS_TO_Y = {'notflip': 0, 'flip': 1}
FRAME_RE = re.compile(r'^(?P<video_id>\d+)_(?P<frame_number>\d+)$')


def setup(seed=42, smoke=False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if smoke: torch.set_num_threads(min(2, torch.get_num_threads()))
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()


def write_json(path, payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding='utf8')


def get_settings(root, smoke=False):
    root=Path(root).resolve()
    # MONREADER_DATA_ROOT is optional; default is the original dataset placed under data/raw/.
    raw=Path(os.environ.get('MONREADER_DATA_ROOT',str(root/'data'/'raw'))).expanduser().resolve()
    out=Path(os.environ.get('MONREADER_OUTPUT_ROOT',str(root/'generated'/'outputs'))).expanduser().resolve()
    return dict(root=str(root),raw=str(raw),out=str(out),smoke=bool(smoke),seed=42,
                image_size=64 if smoke else 224,max_epochs=1 if smoke else 50,
                patience=1 if smoke else 4,qualifying_trials=1 if smoke else 30,
                bootstrap_resamples=40 if smoke else 5000,
                validation_batch_cap=8 if smoke else 32,
                max_batches=1 if smoke else None,
                resnet_pretrained=False if smoke else True,
                tuner_backend=os.getenv('MONREADER_TUNER','optuna'),
                tracking=os.getenv('MONREADER_WANDB','disabled'),
                decision_threshold=0.50)


def load_settings(root):
    p=Path(root)/'generated'/'experiment_settings.json'
    if not p.exists():raise FileNotFoundError('Run Notebook 01 first: '+str(p))
    return json.loads(p.read_text())


def discover_manifest(raw):
    """Strict filename parsing: CLASS/ENVIRONMENT_FRAME.jpg. Never invent identifiers."""
    raw=Path(raw)
    if not raw.exists():raise FileNotFoundError(f'Dataset directory missing: {raw}')
    paths=sorted(p for p in raw.rglob('*') if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.png','.bmp','.webp'))
    if not paths:raise ValueError(f'No readable images under {raw}. Supply the real MonReader dataset.')
    rows=[];bad=[]
    for p in paths:
        rel=p.relative_to(raw).as_posix()
        labels=[part.lower() for part in p.parts if part.lower() in CLASS_TO_Y]
        m=FRAME_RE.fullmatch(p.stem)
        if len(set(labels))!=1 or m is None:
            bad.append(rel);continue
        label=labels[-1]; video=m.group('video_id'); frame=int(m.group('frame_number'))
        # Keep the zero-padded video identifier as it appears in the filename.
        sid=hashlib.sha256(rel.encode()).hexdigest()[:16]
        rows.append(dict(sample_id=sid,relative_path=rel,path=str(p.resolve()),label=label,
                         target=CLASS_TO_Y[label],video_id=video,environment_id=video,
                         frame_number=frame,source_key=f'{label}::{video}'))
    if bad:raise ValueError('Could not infer label/video/frame for these files (first 10): '+str(bad[:10]))
    df=pd.DataFrame(rows).sort_values('relative_path').reset_index(drop=True)
    if df.sample_id.duplicated().any():raise AssertionError('Sample ID collision')
    return df


def audit_manifest(df, out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    # A repeated number across classes is evidence to investigate, not evidence of same physical video.
    overlap=sorted(set(df.loc[df.label=='flip','video_id'])&set(df.loc[df.label=='notflip','video_id']))
    collisions=pd.DataFrame({'shared_numeric_environment_id':overlap})
    collisions.to_csv(out/'cross_label_environment_collisions.csv',index=False)
    dup_frame=df[df.duplicated(['source_key','frame_number'],keep=False)]
    dup_frame.to_csv(out/'repeated_source_frame_numbers.csv',index=False)
    # Decoded-pixel hashes identify exact content duplicates even if filenames differ.
    digests=[];bad=[]
    for row in df.itertuples(index=False):
        try:
            with Image.open(row.path) as im:
                pixels=np.asarray(im.convert('RGB'))
                digest=hashlib.sha256(pixels.shape.__repr__().encode()+pixels.tobytes()).hexdigest()
            digests.append((row.sample_id,digest))
        except Exception as exc:bad.append((row.relative_path,str(exc)))
    if bad:raise ValueError('Unreadable images: '+str(bad[:5]))
    duplicate_pixels=pd.DataFrame(digests,columns=['sample_id','pixel_sha256'])
    duplicate_pixels=duplicate_pixels[duplicate_pixels.duplicated('pixel_sha256',keep=False)]
    duplicate_pixels.to_csv(out/'duplicate_pixels.csv',index=False)
    if len(dup_frame):warnings.warn('Repeated frame numbers detected; inspect audit before accepting source chronology.')
    # Cross-label collisions are NOT automatically leakage; human verification may be needed.
    summary=dict(n_images=len(df),class_counts=df.label.value_counts().to_dict(),
                 n_reconstructed_sources=int(df.source_key.nunique()),n_numeric_ids=int(df.video_id.nunique()),
                 cross_label_numeric_collisions=len(overlap),repeated_source_frames=len(dup_frame),
                 duplicate_content_rows=len(duplicate_pixels))
    write_json(out/'dataset_audit.json',summary)
    return summary


def _finish(parts, regime):
    df=pd.concat(parts,ignore_index=True);df['regime']=regime
    return df.sort_values('sample_id').reset_index(drop=True)


def make_splits(manifest,seed=42):
    # O: 72/8/20 randomized *individual frames*; no source disjointness requirement.
    dev,test=train_test_split(manifest,test_size=.20,random_state=seed,stratify=manifest.label)
    train,val=train_test_split(dev,test_size=.10,random_state=seed,stratify=dev.label)
    three=[a.copy() for a in (train,val,test)]
    for a,r in zip(three,('train','validation','test')):a['role']=r
    O=_finish(three,'O')
    # S: source groups are the units assigned to train/validation/test.
    sources=manifest[['source_key','label']].drop_duplicates().reset_index(drop=True)
    dev_s,test_s=train_test_split(sources,test_size=.20,random_state=seed,stratify=sources.label)
    train_s,val_s=train_test_split(dev_s,test_size=.10,random_state=seed,stratify=dev_s.label)
    mapping={str(k):role for tab,role in [(train_s,'train'),(val_s,'validation'),(test_s,'test')] for k in tab.source_key}
    S=manifest.copy();S['role']=S.source_key.map(mapping);S['regime']='S'
    # T: chronological 80/10/10 within *every* reconstructed source.
    pieces=[]
    for source,g in manifest.groupby('source_key',sort=True):
        t=g.sort_values(['frame_number','sample_id']).copy(); n=len(t)
        if n<3:raise ValueError(f'At least three frames required for temporal source {source}')
        nv=max(1,round(.10*n));nt=max(1,round(.10*n));ntr=n-nv-nt
        if ntr<1:raise ValueError('Insufficient training frames')
        t['role']=['train']*ntr+['validation']*nv+['test']*nt
        pieces.append(t)
    T=_finish(pieces,'T')
    # ST: known sources -> early train/later val; held-out sources -> early context/later test.
    known,unseen=train_test_split(sources,test_size=.20,random_state=seed,stratify=sources.label)
    unseen_keys=set(unseen.source_key);pieces=[]
    for source,g in manifest.groupby('source_key',sort=True):
        t=g.sort_values(['frame_number','sample_id']).copy();n=len(t)
        if n<2:raise ValueError('ST requires >=2 frames per source')
        late=max(1,round(.20*n));early=n-late
        if early<1:raise ValueError('ST needs early source frames')
        if source in unseen_keys:
            t['role']=['context_unseen_early']*early+['test']*late
        else:t['role']=['train']*early+['validation']*late
        pieces.append(t)
    ST=_finish(pieces,'ST')
    result={'O':O,'S':S,'T':T,'ST':ST}
    for regime,df in result.items():validate_split(manifest,df,regime)
    return result


def validate_split(manifest, split, regime):
    original=set(manifest.sample_id)
    if len(split)!=len(manifest) or set(split.sample_id)!=original or split.sample_id.duplicated().any():
        raise AssertionError(f'{regime}: sample accounting failed')
    for r in ('train','validation','test'):
        t=split[split.role==r]
        if len(t)==0 or set(t.label)!={'flip','notflip'}:raise AssertionError(f'{regime}: {r} must include both classes')
    if regime in ('S','ST'):
        tests=set(split.loc[split.role=='test','source_key'])
        dev=set(split.loc[split.role.isin(['train','validation']),'source_key'])
        if tests&dev:raise AssertionError(f'{regime}: unseen-test source leakage')
    if regime=='S':
        if (split.groupby('source_key').role.nunique()>1).any():raise AssertionError('S: source assigned multiple roles')
    if regime=='T':
        for _,g in split.groupby('source_key'):
            tr=g.loc[g.role=='train','frame_number'];val=g.loc[g.role=='validation','frame_number'];te=g.loc[g.role=='test','frame_number']
            if not(tr.max()<val.min() and val.max()<te.min()):raise AssertionError('T chronology violation')
    if regime=='ST':
        for _,g in split.groupby('source_key'):
            if 'train' in set(g.role):
                a=g.loc[g.role=='train','frame_number'];b=g.loc[g.role=='validation','frame_number']
            else:
                a=g.loc[g.role=='context_unseen_early','frame_number'];b=g.loc[g.role=='test','frame_number']
            if not(a.max()<b.min()):raise AssertionError('ST chronology violation')


def save_splits(splits,out):
    p=Path(out)/'splits';p.mkdir(parents=True,exist_ok=True);rows=[]
    for regime,df in splits.items():
        f=p/f'{regime}_split.csv';df.to_csv(f,index=False)
        vc=df.role.value_counts();rows.append(dict(regime=regime,**{k:int(vc.get(k,0)) for k in ['train','validation','test','context_unseen_early']},sha256=file_sha256(f)))
    summary=pd.DataFrame(rows);summary.to_csv(Path(out)/'split_summary.csv',index=False)
    return summary


def plot_regime_examples(splits,filename,seed=42):
    rng=random.Random(seed);fig,axes=plt.subplots(4,4,figsize=(13,11))
    roles={'O':['train','validation','test','train'],'S':['train','validation','test','test'],
           'T':['train','validation','test','test'],'ST':['train','validation','context_unseen_early','test']}
    for i,regime in enumerate(REGIMES):
        frame=splits[regime]
        for j,role in enumerate(roles[regime]):
            subset=frame[frame.role==role];row=subset.iloc[rng.randrange(len(subset))]
            with Image.open(row.path) as im:img=im.convert('RGB')
            ax=axes[i,j];ax.imshow(img);ax.set_xticks([]);ax.set_yticks([])
            ax.set_title(f'{role}\nVideo {row.video_id} / frame {row.frame_number}',fontsize=8)
        axes[i,0].set_ylabel(regime,rotation=0,labelpad=28,fontweight='bold')
    fig.suptitle('Evaluation Regime Examples (sampled from actual split manifests)',fontsize=13)
    fig.tight_layout();filename=Path(filename);filename.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(filename,dpi=140,bbox_inches='tight');plt.close(fig)
    return filename


class FrameDataset(Dataset):
    def __init__(self,table,image_size=224,gray=False,imagenet=False):
        self.table=table.reset_index(drop=True);self.gray=gray
        t=[transforms.Resize((image_size,image_size)),transforms.ToTensor()]
        if imagenet:t.append(transforms.Normalize([.485,.456,.406],[.229,.224,.225]))
        self.transform=transforms.Compose(t)
    def __len__(self):return len(self.table)
    def __getitem__(self,i):
        row=self.table.iloc[i]
        with Image.open(row.path) as im:
            if self.gray:
                # 1-channel intensities replicated to three channels to preserve architecture and shape.
                im=im.convert('L').convert('RGB')
            else:im=im.convert('RGB')
            x=self.transform(im)
        return x,torch.tensor(float(row.target),dtype=torch.float32)


def make_loader(table,settings,batch_size=16,shuffle=False,gray=False,imagenet=False):
    return DataLoader(FrameDataset(table,settings['image_size'],gray,imagenet),batch_size=min(int(batch_size),len(table)),shuffle=shuffle,num_workers=0,pin_memory=False)


class ScratchCNN(nn.Module):
    def __init__(self,depth=3,starting_filters=16,dropout=.2):
        super().__init__(); layers=[];nin=3;nout=starting_filters
        for _ in range(int(depth)):
            # The cap prevents exponentially expanding channels from requiring implausible VRAM.
            nout=min(int(nout),256)
            layers.extend([nn.Conv2d(nin,nout,3,padding=1),nn.BatchNorm2d(nout),nn.ReLU(inplace=True),nn.MaxPool2d(2)])
            nin=nout;nout*=2
        self.features=nn.Sequential(*layers)
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Sequential(nn.Dropout(float(dropout)),nn.Linear(nin,1))
    def forward(self,x):return self.head(self.pool(self.features(x)).flatten(1)).flatten()


def get_resnet(pretrained=True):
    model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc=nn.Linear(model.fc.in_features,1)
    return model


def build_optimizer(model,c):
    name=c['optimizer'].lower(); kw={'lr':float(c['learning_rate']),'weight_decay':float(c['weight_decay'])}
    if name=='adam':return torch.optim.Adam(model.parameters(),**kw)
    if name=='adamw':return torch.optim.AdamW(model.parameters(),**kw)
    if name=='rmsprop':return torch.optim.RMSprop(model.parameters(),**kw)
    if name=='sgd':return torch.optim.SGD(model.parameters(),momentum=.9,**kw)
    raise ValueError(name)


def predict_frame_table(model,table,settings,device,batch_size=32,gray=False,imagenet=False):
    loader=make_loader(table,settings,min(batch_size,settings['validation_batch_cap']),False,gray,imagenet)
    model.eval();prob=[]
    with torch.no_grad():
        for images,_ in loader:
            logits=model(images.to(device)).flatten();prob.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    result=table[['sample_id','relative_path','label','target','source_key','video_id','frame_number']].copy().reset_index(drop=True)
    result['prob_flip']=np.asarray(prob,dtype=float)
    result['prediction']=(result.prob_flip>=settings['decision_threshold']).astype(int)
    result['correct']=(result.prediction==result.target)
    return result


def performance(predictions):
    y=np.asarray(predictions.target,dtype=int);p=np.asarray(predictions.prob_flip,dtype=float)
    yp=(p>=.50).astype(int)
    d=dict(n_images=int(len(y)),accuracy=float(accuracy_score(y,yp)),
           balanced_accuracy=float(balanced_accuracy_score(y,yp)),
           macro_f1=float(f1_score(y,yp,average='macro',labels=[0,1],zero_division=0)),
           flip_precision=float(precision_score(y,yp,pos_label=1,zero_division=0)),
           flip_recall=float(recall_score(y,yp,pos_label=1,zero_division=0)))
    if len(np.unique(y))==2:
        d['roc_auc_flip']=float(roc_auc_score(y,p));d['pr_auc_flip']=float(average_precision_score(y,p))
    else:d['roc_auc_flip']=float('nan');d['pr_auc_flip']=float('nan')
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    d.update(tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))
    return d


def fit_model(model,train,validation,settings,config,device,gray=False,imagenet=False,log_fn=None):
    model.to(device);optimizer=build_optimizer(model,config)
    loss_fn=nn.BCEWithLogitsLoss()
    trainloader=make_loader(train,settings,config['batch_size'],True,gray,imagenet)
    valload=make_loader(validation,settings,min(config['batch_size'],settings['validation_batch_cap']),False,gray,imagenet)
    best=-1.0;best_epoch=0;best_state=None;stale=0;hist=[]
    for epoch in range(1,settings['max_epochs']+1):
        model.train();losses=[]
        for bi,(images,targets) in enumerate(trainloader):
            logits=model(images.to(device)).flatten();loss=loss_fn(logits,targets.to(device))
            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            losses.append(float(loss.item()))
            if settings.get('max_batches') is not None and bi+1>=settings['max_batches']:break
        model.eval();vprobs=[];vy=[];vloss=[]
        with torch.no_grad():
            for vi,(images,targets) in enumerate(valload):
                logits=model(images.to(device)).flatten();vloss.append(float(loss_fn(logits,targets.to(device)).item()))
                vprobs.extend(torch.sigmoid(logits).cpu().numpy().tolist());vy.extend(targets.long().numpy().tolist())
                if settings.get('max_batches') is not None and vi+1>=settings['max_batches']:break
        score=float(f1_score(vy,(np.array(vprobs)>=.50).astype(int),average='macro',labels=[0,1],zero_division=0))
        row=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_loss=float(np.mean(vloss)),validation_macro_f1=score)
        hist.append(row)
        if log_fn is not None:log_fn(row)
        if score>best: # strict > selects first epoch in a tie
            best=score;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict());stale=0
        else:stale+=1
        if stale>=settings['patience']:break
    if best_state is None:raise RuntimeError('No checkpoint selected')
    model.load_state_dict(best_state)
    return model,pd.DataFrame(hist),dict(best_epoch=int(best_epoch),best_validation_macro_f1=float(best))


def trial_config(trial,smoke=False):
    if smoke:
        # Full trial/tuning/training code is executed in smoke mode; resource-limited parameters only.
        return dict(optimizer=trial.suggest_categorical('optimizer',['adam','adamw','sgd','rmsprop']),
                    learning_rate=trial.suggest_float('learning_rate',1e-4,2e-3,log=True),
                    batch_size=trial.suggest_int('batch_size',8,16),
                    weight_decay_mode=trial.suggest_categorical('weight_decay_mode',['zero','positive']),
                    positive_weight_decay=trial.suggest_float('positive_weight_decay',1e-6,1e-3,log=True),
                    dropout=trial.suggest_float('dropout',.0,.3),
                    depth=trial.suggest_int('depth',1,2),starting_filters=trial.suggest_int('starting_filters',8,12))
    return dict(optimizer=trial.suggest_categorical('optimizer',['adam','adamw','sgd','rmsprop']),
                learning_rate=trial.suggest_float('learning_rate',1e-5,1e-2,log=True),
                batch_size=trial.suggest_int('batch_size',8,256),
                weight_decay_mode=trial.suggest_categorical('weight_decay_mode',['zero','positive']),
                positive_weight_decay=trial.suggest_float('positive_weight_decay',1e-6,1e-2,log=True),
                dropout=trial.suggest_float('dropout',0.,.5),
                depth=trial.suggest_int('depth',1,7),starting_filters=trial.suggest_int('starting_filters',8,64))


def resolve_config(c):
    x=dict(c);x['weight_decay']=0. if x.get('weight_decay_mode')=='zero' else float(x['positive_weight_decay'])
    return x


def tune_one_regime(split,regime,settings,device):
    """Independent fresh Bayesian TPE search, validation-only model selection."""
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    out=Path(settings['out'])/'scratch'/regime;out.mkdir(parents=True,exist_ok=True)
    train=split[split.role=='train'];val=split[split.role=='validation'];test=split[split.role=='test']
    study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=settings['seed']))
    trials=[];target=settings['qualifying_trials'];attempts=0;qualified=0
    best_model=None;best_info=None;best_config=None;best_history=None
    while qualified<target and attempts<max(5*target,3):
        trial=study.ask();attempts+=1;config=resolve_config(trial_config(trial,settings['smoke']))
        try:
            setup(settings['seed']+attempts,settings['smoke'])
            model=ScratchCNN(config['depth'],config['starting_filters'],config['dropout'])
            model,hist,info=fit_model(model,train,val,settings,config,device)
            v=float(info['best_validation_macro_f1'])
            # On a smoke fixture a zero score is an executed test, not a qualified research trial.
            qualifies=np.isfinite(v) and v>0
            study.tell(trial,v)
            if qualifies or settings['smoke']:qualified+=1
            records=dict(trial=trial.number,validation_macro_f1=v,qualifies=bool(qualifies),status='finished',**config,**info)
            trials.append(records)
            hist.to_csv(out/f'trial_{trial.number:03d}_epoch_history.csv',index=False)
            if best_info is None or v>best_info['best_validation_macro_f1']:
                best_model=copy.deepcopy(model).cpu();best_info=info;best_config=config;best_history=hist
        except (RuntimeError,MemoryError) as exc:
            study.tell(trial,state=optuna.trial.TrialState.FAIL)
            trials.append(dict(trial=trial.number,status='failed',reason=str(exc)[:300]))
            if torch.cuda.is_available():torch.cuda.empty_cache()
    if best_model is None:raise RuntimeError(f'No valid models trained in regime {regime}')
    if qualified<target:raise RuntimeError(f'{regime}: only {qualified}/{target} qualifying trials; cannot silently claim target reached')
    pd.DataFrame(trials).to_csv(out/'bayesian_trials.csv',index=False)
    best_history.to_csv(out/'selected_training_history.csv',index=False)
    torch.save({'architecture':'scratch','config':best_config,'weights':best_model.state_dict(),
                'selection_metric':'validation_macro_f1','best_epoch':best_info['best_epoch']},out/'selected_checkpoint.pt')
    write_json(out/'selected_config.json',dict(config=best_config,**best_info,qualified=qualified,attempts=attempts))
    best_model.to(device)
    valpred=predict_frame_table(best_model,val,settings,device,best_config['batch_size'])
    testpred=predict_frame_table(best_model,test,settings,device,best_config['batch_size'])
    valpred.to_csv(out/'validation_predictions.csv',index=False)
    testpred.to_csv(out/'heldout_test_predictions.csv',index=False)
    # Distinguish checkpoint selection from heldout reporting, in names and files.
    met={**{f'val_{k}':v for k,v in performance(valpred).items()},
         **{f'test_{k}':v for k,v in performance(testpred).items()},'regime':regime,'model_family':'scratch'}
    write_json(out/'metrics.json',met)
    return met


def train_resnet_regime(split,regime,settings,device):
    out=Path(settings['out'])/'resnet18'/regime;out.mkdir(parents=True,exist_ok=True)
    setup(settings['seed'],settings['smoke']);model=get_resnet(settings['resnet_pretrained'])
    config={'optimizer':'adamw','learning_rate':1e-4,'weight_decay':1e-4,'batch_size':8 if settings['smoke'] else 32}
    tr=split[split.role=='train'];val=split[split.role=='validation'];te=split[split.role=='test']
    model,hist,info=fit_model(model,tr,val,settings,config,device,imagenet=True)
    hist.to_csv(out/'training_history.csv',index=False)
    torch.save({'architecture':'resnet18','weights':model.cpu().state_dict(),'config':config,
                'best_epoch':info['best_epoch'],'pretrained_start':settings['resnet_pretrained']},out/'selected_checkpoint.pt')
    model.to(device)
    valpred=predict_frame_table(model,val,settings,device,config['batch_size'],imagenet=True)
    testpred=predict_frame_table(model,te,settings,device,config['batch_size'],imagenet=True)
    valpred.to_csv(out/'validation_predictions.csv',index=False);testpred.to_csv(out/'heldout_test_predictions.csv',index=False)
    m={**{f'val_{k}':v for k,v in performance(valpred).items()},
       **{f'test_{k}':v for k,v in performance(testpred).items()},'regime':regime,'model_family':'resnet18',**info}
    write_json(out/'metrics.json',m)
    return m


def train_grayscale(split,regime,settings,device):
    out=Path(settings['out'])/'grayscale'/regime;out.mkdir(parents=True,exist_ok=True)
    cfg=json.loads((Path(settings['out'])/'scratch'/regime/'selected_config.json').read_text())['config']
    setup(settings['seed'],settings['smoke'])
    model=ScratchCNN(cfg['depth'],cfg['starting_filters'],cfg['dropout'])
    tr=split[split.role=='train'];val=split[split.role=='validation'];te=split[split.role=='test']
    model,hist,info=fit_model(model,tr,val,settings,cfg,device,gray=True)
    hist.to_csv(out/'training_history.csv',index=False)
    torch.save({'architecture':'scratch_grayscale','config':cfg,'weights':model.cpu().state_dict()},out/'checkpoint.pt')
    model.to(device)
    pred=predict_frame_table(model,te,settings,device,cfg['batch_size'],gray=True)
    pred.to_csv(out/'heldout_test_predictions.csv',index=False)
    m={**performance(pred),'regime':regime,'model_family':'scratch_grayscale',**info}
    write_json(out/'metrics.json',m);return m


def plot_model_comparison(results,out):
    p=pd.DataFrame(results);fig,ax=plt.subplots(figsize=(10,5));xs=np.arange(4);width=.36
    for i,(family,label) in enumerate([('scratch','Scratch CNN'),('resnet18','Pretrained ResNet18')]):
        vals=[float(p.loc[(p.regime==r)&(p.model_family==family),'test_macro_f1'].iloc[0]) for r in REGIMES]
        bars=ax.bar(xs+(i-.5)*width,vals,width,label=label)
        for b,v in zip(bars,vals):ax.text(b.get_x()+b.get_width()/2,v+.015,f'{v:.3f}',ha='center',fontsize=9)
    ax.set_xticks(xs,labels=['O\nOriginal','S\nSource-safe','T\nTemporal','ST\nSource + Temporal']);ax.set_ylim(0,1.12)
    ax.set_ylabel('Held-out Macro F1');ax.set_title('Model Comparison Across the Four Evaluation Regimes');ax.legend();ax.grid(axis='y',alpha=.25)
    fig.tight_layout();out=Path(out);out.parent.mkdir(parents=True,exist_ok=True);fig.savefig(out,dpi=160);plt.close(fig)
    return out


def plot_grayscale_examples(manifest,out,seed=42):
    fig,axes=plt.subplots(2,2,figsize=(8,9))
    for i,label in enumerate(('flip','notflip')):
        row=manifest[manifest.label==label].sort_values('sample_id').iloc[0]
        with Image.open(row.path) as im:rgb=np.asarray(im.convert('RGB'));gray=np.asarray(im.convert('L'))
        axes[i,0].imshow(rgb);axes[i,1].imshow(gray,cmap='gray',vmin=0,vmax=255)
        axes[i,0].set_ylabel(label,fontweight='bold');
        for j in range(2):axes[i,j].set_xticks([]);axes[i,j].set_yticks([])
    axes[0,0].set_title('RGB');axes[0,1].set_title('Grayscale')
    fig.suptitle('Same original frame before and after grayscale conversion')
    fig.tight_layout();out=Path(out);out.parent.mkdir(parents=True,exist_ok=True);fig.savefig(out,dpi=150);plt.close(fig)
    return out


def source_bootstrap(pred,seed=42,reps=5000):
    rng=np.random.default_rng(seed);groups=[d for _,d in pred.groupby('source_key',sort=True)]
    if not groups:raise ValueError('No sources')
    scores=np.empty(reps)
    for i in range(reps):
        sampled=rng.integers(0,len(groups),len(groups));data=pd.concat([groups[k] for k in sampled],ignore_index=True)
        scores[i]=performance(data)['macro_f1']
    return dict(point_macro_f1=performance(pred)['macro_f1'],ci_lower=float(np.percentile(scores,2.5)),
                ci_upper=float(np.percentile(scores,97.5)),bootstrap_mean=float(scores.mean()),
                n_sources=len(groups),n_resamples=reps),scores


def paired_source_bootstrap(rgb,gray,seed=42,reps=5000):
    a=rgb.merge(gray[['sample_id','prob_flip']],on='sample_id',suffixes=('_rgb','_gray'),validate='one_to_one')
    if len(a)!=len(rgb) or len(a)!=len(gray):raise AssertionError('RGB/grayscale predictions are not exactly paired')
    groups=[d for _,d in a.groupby('source_key',sort=True)];rng=np.random.default_rng(seed);deltas=[]
    for _ in range(reps):
        b=pd.concat([groups[k] for k in rng.integers(0,len(groups),len(groups))])
        p_rgb=pd.DataFrame(dict(target=b.target,prob_flip=b.prob_flip_rgb))
        p_gray=pd.DataFrame(dict(target=b.target,prob_flip=b.prob_flip_gray))
        deltas.append(performance(p_gray)['macro_f1']-performance(p_rgb)['macro_f1'])
    return dict(point_delta=performance(gray)['macro_f1']-performance(rgb)['macro_f1'],
                ci_lower=float(np.percentile(deltas,2.5)),ci_upper=float(np.percentile(deltas,97.5)))


def classify_errors(pred,high_conf=.90):
    met=performance(pred);incorrect=pred[pred.target!=pred.prediction]
    confidence=np.where(incorrect.prediction==1,incorrect.prob_flip,1-incorrect.prob_flip)
    return dict(n_errors=int(len(incorrect)),error_rate=float(len(incorrect)/len(pred)),fp=met['fp'],fn=met['fn'],
                high_confidence_errors=int((confidence>=high_conf).sum()),
                high_confidence_error_share=float((confidence>=high_conf).mean()) if len(incorrect) else 0.)


def load_resnet_checkpoint(settings,regime,device):
    p=Path(settings['out'])/'resnet18'/regime/'selected_checkpoint.pt'
    ckpt=torch.load(p,map_location='cpu',weights_only=False)
    model=get_resnet(pretrained=False);model.load_state_dict(ckpt['weights']);model.to(device);model.eval()
    return model


def cam_for_image(model,image_path,device,image_size,predicted_class):
    """Binary one-logit predicted-class Grad-CAM: flip=+logit; notflip=-logit."""
    with Image.open(image_path) as im:rgb=np.asarray(im.convert('RGB').resize((image_size,image_size)))
    x=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])(Image.fromarray(rgb)).unsqueeze(0).to(device)
    activations=[]
    handle=model.layer4[-1].conv2.register_forward_hook(lambda _m,_inp,out:activations.append(out))
    with torch.enable_grad():
        out=model(x);activation=activations[0];score=out.flatten()[0]*(1 if predicted_class==1 else -1)
        gradients=torch.autograd.grad(score,activation,retain_graph=False)[0]
        weights=gradients.mean(dim=(2,3),keepdim=True)
        cam=torch.relu((weights*activation).sum(dim=1,keepdim=True))
        cam=torch.nn.functional.interpolate(cam,size=(image_size,image_size),mode='bilinear',align_corners=False)[0,0]
        heat=cam.detach().cpu().numpy()
    handle.remove()
    heat=heat-heat.min();heat=heat/(heat.max()+1e-8)
    from matplotlib import colormaps
    color=colormaps['jet'](heat)[...,:3]
    overlay=(.45*rgb.astype(float)/255+.55*color).clip(0,1)
    return overlay,heat


def select_cam_cases(pred):
    p=pred.copy();p['is_error']=p.prediction!=p.target
    p['confidence']=np.where(p.prediction==1,p.prob_flip,1-p.prob_flip)
    candidates=[('Correct notflip',p[(~p.is_error)&(p.target==0)].sort_values(['confidence','sample_id'],ascending=[False,True])),
                ('Correct flip',p[(~p.is_error)&(p.target==1)].sort_values(['confidence','sample_id'],ascending=[False,True])),
                ('High-confidence error',p[p.is_error].sort_values(['confidence','sample_id'],ascending=[False,True])),
                ('Borderline error',p[p.is_error].assign(dist=lambda x:abs(x.prob_flip-.5)).sort_values(['dist','sample_id']))]
    selected=[];used=set()
    for role,df in candidates:
        hit=next((x for x in df.itertuples(index=False) if x.sample_id not in used),None)
        if hit is None:selected.append(dict(role=role,found=False,sample_id=None))
        else:selected.append(dict(role=role,found=True,sample_id=hit.sample_id,relative_path=hit.relative_path,
                                  target=int(hit.target),prob_flip=float(hit.prob_flip),prediction=int(hit.prediction)))
        if hit is not None:used.add(hit.sample_id)
    return selected


def plot_cam_matrix(settings,device,out):
    fig,axes=plt.subplots(4,4,figsize=(12,13));registry=[]
    for i,regime in enumerate(REGIMES):
        model=load_resnet_checkpoint(settings,regime,device)
        pred=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
        picks=select_cam_cases(pred)
        for j,case in enumerate(picks):
            ax=axes[i,j]
            if case['found']:
                path=Path(settings['raw'])/case['relative_path']
                overlay,heat=cam_for_image(model,path,device,settings['image_size'],case['prediction'])
                ax.imshow(overlay);case['cam_max']=float(heat.max())
            else:ax.text(.5,.5,'No eligible case\nin this test set',ha='center',va='center',transform=ax.transAxes)
            ax.set_xticks([]);ax.set_yticks([])
            if i==0:ax.set_title(case['role'],fontsize=9)
            if j==0:ax.set_ylabel(regime,rotation=0,labelpad=20,fontweight='bold')
            registry.append({'regime':regime,**case})
    fig.suptitle('ResNet18 Grad-CAM: correct and error cases',fontsize=15)
    fig.tight_layout(rect=[0,0,1,.97]);out=Path(out);out.parent.mkdir(parents=True,exist_ok=True);fig.savefig(out,dpi=150);plt.close(fig)
    pd.DataFrame(registry).to_csv(out.parent/'gradcam_16_panel_selection.csv',index=False)
    return out


def tune_one_regime_wandb(split,regime,settings,device):
    """Optional true W&B Bayesian sweep. Requires the researcher's W&B account.

    This is intentionally a NEW sweep; it neither retrieves nor modifies the
    author's earlier W&B experiments. Test data are evaluated only after the
    selected validation winner has been frozen.
    """
    try:import wandb
    except ImportError as exc:raise ImportError('Install wandb and authenticate before enabling W&B Bayesian sweeps') from exc
    if settings['smoke']:raise RuntimeError('Use the offline Optuna path for synthetic smoke testing.')
    out=Path(settings['out'])/'scratch'/regime;out.mkdir(parents=True,exist_ok=True)
    tr=split[split.role=='train'];val=split[split.role=='validation'];te=split[split.role=='test']
    sweep_cfg=dict(method='bayes',metric=dict(name='validation_macro_f1',goal='maximize'),
        parameters=dict(optimizer=dict(values=['adam','adamw','sgd','rmsprop']),
            learning_rate=dict(distribution='log_uniform_values',min=1e-5,max=1e-2),
            batch_size=dict(distribution='int_uniform',min=8,max=256),
            weight_decay_mode=dict(values=['zero','positive']),
            positive_weight_decay=dict(distribution='log_uniform_values',min=1e-6,max=1e-2),
            dropout=dict(distribution='uniform',min=0.,max=.5),
            depth=dict(distribution='int_uniform',min=1,max=7),
            starting_filters=dict(distribution='int_uniform',min=8,max=64)))
    project='monreader-scratch-'+regime
    sweep_id=wandb.sweep(sweep_cfg,project=project)
    rows=[]; state={'attempts':0,'qualifying':0,'best':-1.,'model':None,'config':None,'info':None,'history':None}
    def do_train():
        with wandb.init(project=project,group=regime,job_type='new_bayesian_trial') as run:
            state['attempts']+=1
            c=resolve_config(dict(run.config))
            setup(settings['seed']+state['attempts'],False)
            try:
                model=ScratchCNN(c['depth'],c['starting_filters'],c['dropout'])
                model,hist,info=fit_model(model,tr,val,settings,c,device,log_fn=lambda rec:wandb.log(rec))
                v=info['best_validation_macro_f1']
                qualified=bool(np.isfinite(v) and v>0)
                if qualified:state['qualifying']+=1
                rows.append(dict(attempt=state['attempts'],run_id=run.id,qualifies=qualified,validation_macro_f1=v,**c,**info))
                wandb.log(dict(best_validation_macro_f1=v,qualified=int(qualified)))
                hist.to_csv(out/f'trial_{state["attempts"]:03d}_epoch_history.csv',index=False)
                if v>state['best']:
                    state.update(best=v,model=copy.deepcopy(model).cpu(),config=c,info=info,history=hist)
            except RuntimeError as exc:
                rows.append(dict(attempt=state['attempts'],run_id=run.id,status='failed',reason=str(exc)[:300]))
                if torch.cuda.is_available():torch.cuda.empty_cache()
                raise
    while state['qualifying']<settings['qualifying_trials'] and state['attempts']<settings['qualifying_trials']*5:
        wandb.agent(sweep_id,function=do_train,count=1,project=project)
    if state['qualifying']<settings['qualifying_trials'] or state['model'] is None:
        raise RuntimeError(f'W&B sweep has insufficient qualifying trials for {regime}')
    pd.DataFrame(rows).to_csv(out/'bayesian_trials.csv',index=False)
    state['history'].to_csv(out/'selected_training_history.csv',index=False)
    torch.save(dict(architecture='scratch',config=state['config'],weights=state['model'].state_dict(),
                    selection_metric='validation_macro_f1',best_epoch=state['info']['best_epoch']),out/'selected_checkpoint.pt')
    write_json(out/'selected_config.json',dict(config=state['config'],**state['info'],
                                                qualifying=state['qualifying'],attempts=state['attempts']))
    model=state['model'].to(device)
    valpred=predict_frame_table(model,val,settings,device,state['config']['batch_size'])
    testpred=predict_frame_table(model,te,settings,device,state['config']['batch_size'])
    valpred.to_csv(out/'validation_predictions.csv',index=False)
    testpred.to_csv(out/'heldout_test_predictions.csv',index=False)
    met={**{f'val_{k}':v for k,v in performance(valpred).items()},
         **{f'test_{k}':v for k,v in performance(testpred).items()},'regime':regime,'model_family':'scratch'}
    write_json(out/'metrics.json',met)
    return met


def select_21_interpretability_cases(settings):
    """Choose cases from frozen TEST predictions before viewing images/heatmaps."""
    rows=[];seen=set()
    # Five cases per regime and an extra non-duplicate borderline case if available.
    for regime in REGIMES:
        p=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
        p['error']=p.prediction.ne(p.target)
        p['confidence']=np.where(p.prediction==1,p.prob_flip,1-p.prob_flip)
        p['distance']=abs(p.prob_flip-.5)
        pools=[('correct_notflip',p[(~p.error)&(p.target==0)].sort_values(['confidence','sample_id'],ascending=[False,True])),
               ('correct_flip',p[(~p.error)&(p.target==1)].sort_values(['confidence','sample_id'],ascending=[False,True])),
               ('highest_conf_error',p[p.error].sort_values(['confidence','sample_id'],ascending=[False,True])),
               ('near_threshold_error',p[p.error].sort_values(['distance','sample_id'])),
               ('near_threshold_correct',p[~p.error].sort_values(['distance','sample_id']))]
        for role,pool in pools:
            chosen=next((r for r in pool.itertuples(index=False) if (regime,r.sample_id) not in seen),None)
            if chosen is not None:
                seen.add((regime,chosen.sample_id));rows.append(dict(regime=regime,selection_role=role,
                   sample_id=chosen.sample_id,relative_path=chosen.relative_path,target=int(chosen.target),
                   predicted=int(chosen.prediction),prob_flip=float(chosen.prob_flip)))
    p=pd.read_csv(Path(settings['out'])/'resnet18'/'O'/'heldout_test_predictions.csv')
    p['distance']=abs(p.prob_flip-.5)
    extra=next((r for r in p.sort_values(['distance','sample_id']).itertuples(index=False) if ('O',r.sample_id) not in seen),None)
    if extra is not None:
        rows.append(dict(regime='O',selection_role='additional_borderline',sample_id=extra.sample_id,
                         relative_path=extra.relative_path,target=int(extra.target),predicted=int(extra.prediction),prob_flip=float(extra.prob_flip)))
    return pd.DataFrame(rows)


def generate_activation_diagnostics(settings,device,filename):
    """Save one mean-activation map for each of four ResNet blocks per selected frame."""
    registry=select_21_interpretability_cases(settings)
    output=Path(filename);output.parent.mkdir(parents=True,exist_ok=True)
    registry.to_csv(output.parent/'interpretability_case_registry.csv',index=False)
    diag=[];preview=[]
    for regime in REGIMES:
        model=load_resnet_checkpoint(settings,regime,device)
        selected=registry[registry.regime==regime]
        if settings['smoke']:selected=selected.head(1)
        for row in selected.itertuples(index=False):
            with Image.open(Path(settings['raw'])/row.relative_path) as im:im=im.convert('RGB').resize((settings['image_size'],settings['image_size']))
            x=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])(im).unsqueeze(0).to(device)
            captures={};handles=[]
            for idx in (1,2,3,4):
                layer=getattr(model,f'layer{idx}')
                handles.append(layer.register_forward_hook(lambda m,inp,out,i=idx:captures.__setitem__(i,out.detach().cpu())))
            with torch.no_grad():model(x)
            for h in handles:h.remove()
            means=[]
            for idx in (1,2,3,4):
                fmap=captures[idx][0].abs().mean(0).numpy();fmap=fmap-fmap.min();fmap/=fmap.max()+1e-8
                means.append(fmap)
                map_path=output.parent/'activation_maps'/f'{regime}_{row.sample_id}_layer{idx}.png'
                map_path.parent.mkdir(parents=True,exist_ok=True)
                plt.imsave(map_path,fmap,cmap='viridis')
                diag.append(dict(regime=regime,sample_id=row.sample_id,layer=idx,
                                 mean_abs_activation=float(captures[idx].abs().mean()),map_path=str(map_path)))
            if len(preview)<4:preview.append((regime,means))
    pd.DataFrame(diag).to_csv(output.parent/'activation_summary.csv',index=False)
    fig,axes=plt.subplots(max(1,len(preview)),4,figsize=(11,2.6*max(1,len(preview))))
    axes=np.asarray(axes).reshape(max(1,len(preview)),4)
    for ri,(regime,maps) in enumerate(preview):
        for col,activation in enumerate(maps):
            axes[ri,col].imshow(activation,cmap='viridis');axes[ri,col].axis('off')
            if ri==0:axes[ri,col].set_title(f'ResNet layer {col+1}')
            if col==0:axes[ri,col].text(-.12,.5,regime,transform=axes[ri,col].transAxes,va='center',fontweight='bold')
    fig.suptitle('ResNet18 Intermediate Activation Progression',y=1.01)
    fig.tight_layout();fig.savefig(output,dpi=140,bbox_inches='tight');plt.close(fig)
    return registry,pd.DataFrame(diag)


In [ ]:
sys.path.insert(0, str(GENERATED))
import monreader_methods as mr
settings = mr.get_settings(ROOT, smoke=SMOKE_TEST)
mr.setup(settings['seed'],SMOKE_TEST)
mr.write_json(GENERATED/'experiment_settings.json',settings)
print({k:settings[k] for k in ['raw','out','smoke','image_size','max_epochs','patience','qualifying_trials']})

## Inspect and decode the original filenames

A name such as `0065_000000015.jpg` contains two candidate fields: recording/environment **0065** and frame index **15**. The labels are obtained from the actual enclosing `flip`/`notflip` directory, not guessed from pixels. Numeric identifiers can recur across classes; cross-label collisions are reported, and `label::video_id` is treated as a reconstructed grouping hypothesis, subject to verification against recording provenance. Invalid filename patterns cause a hard failure rather than silently dropping images.

In [ ]:
from IPython.display import display
import pandas as pd
manifest = mr.discover_manifest(settings['raw'])
display(manifest[['relative_path','label','video_id','frame_number','source_key']].head(12))
audit = mr.audit_manifest(manifest, settings['out'])
display(pd.DataFrame([audit]).T.rename(columns={0:'observed'}))
manifest.to_csv(Path(settings['out'])/'canonical_manifest.csv',index=False)

## Evaluate source structure and chronological quality

We now inspect **measured** class counts and frames per reconstructed source. A large number of frames per source implies correlated observations; it does not by itself establish duplicated pixels. The collision and duplicate-pixel reports were calculated directly above. The temporal split will reject sources with insufficient frames or repeated chronology where strict ordering cannot be verified.

In [ ]:
import matplotlib.pyplot as plt
frames_per_source=manifest.groupby('source_key').size().rename('frames')
display(pd.DataFrame({'class_frames':manifest.label.value_counts()}))
display(frames_per_source.describe().to_frame())
fig,ax=plt.subplots(figsize=(7,3.5))
ax.hist(frames_per_source.to_numpy(),bins=min(20,len(frames_per_source)))
ax.set(title='Observed frames per reconstructed source',xlabel='Frames',ylabel='Number of sources')
fig.tight_layout()
fig.savefig(Path(settings['out'])/'figures'/'frames_per_source.png',dpi=150) if (Path(settings['out'])/'figures').mkdir(parents=True,exist_ok=True) is None else None
plt.show()

## Construct O, S, T and ST without looking at model performance

**O** randomly stratifies frames into approximately 72% training, 8% validation and 20% final testing. **S** makes source-disjoint partitions. **T** assigns early/middle/late frames chronologically within every source. **ST** separates unseen sources first; their early frames are explicitly labelled `context_unseen_early` and are never used for fitting or selection. Exact counts are derived, **not entered as targets**. Tests verify sample accounting, class coverage and the restrictions appropriate to each regime. Every split is saved and SHA-256 fingerprinted before training.

In [ ]:
splits = mr.make_splits(manifest,seed=settings['seed'])
split_summary = mr.save_splits(splits,settings['out'])
display(split_summary)
for regime, df in splits.items():
    print(f'\n{regime} — class distribution')
    display(pd.crosstab(df.role,df.label))

## Regime figure and research handoff

Real image panels are selected by their actual assigned roles. The figure is **illustrative** of the four designs, not a second source of truth for split membership. Notebooks 02–05 will consume the saved manifest and split CSVs, so the entire experiment can be reopened in a new Jupyter kernel.

In [ ]:
figfile = mr.plot_regime_examples(splits,Path(settings['out'])/'figures'/'evaluation_regime_examples.png')
from IPython.display import Image as IPyImage,display as show
show(IPyImage(filename=str(figfile)))
print('Saved:',figfile)
print('Notebook 01 finished: derived, checked, and fingerprinted all split manifests.')